In [1]:
import numpy as np
import pandas as pd
import requests
import itertools
from math import ceil
from time import sleep
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Risk Free Rate
RF = 0.04
CONF_LEVEL = 0.95

In [2]:
def get_alphavantagekey(path):
    with open(path) as f:
        key  = f.read().strip()
    return key
key=get_alphavantagekey('Keys/alphavantage.txt')

# **Securities**

In [3]:
class Security:
    def __init__(self, name, identifier, mu, sigma, n, rf, conf_level):
        self.name = name
        self.identifier = identifier
        self.mu = mu
        self.sigma = sigma
        self.n = n
        self.rf = rf
        self.conf_level = conf_level
        self.generate_returns(self.mu, self.sigma, self.n)
        self.calc_max_drawdown()
        self.calc_CVaR(conf_level)
        self.calc_sharpe_ratio(rf)

    def __str__(self):
        return f"{self.name}\n{'='*50}\nidentifier:{self.identifier}\nReturns:{len(self.returns)}\nMean:{self.mu*100:.2f}%\nStandard Deviation:{self.sigma*100:.2f}%\nSharpe Ratio:{self.sharpe_ratio:.2f}\nCVaR:{self.CVaR*100:.2f}%\nMax Drawdown:{self.MDD:.2f}%"

    def calc_sharpe_ratio(self, rf):
        sr = round((self.mu  - rf) / self.sigma,2)
        self.sharpe_ratio = sr
        return sr

    def calc_CVaR(self, conf_level):
        """
        conditional value at risk for a confidence interval
        Find the average of the worst returns (below the conf level)
        """
        # number of returns
        n = len(self.returns)

        # sort the returns
        r = np.sort(self.returns)

        # number of returns lower than the confidence level
        num_below_conf = int((1- conf_level) * n)

        # average of the worst returns (below the conf level)
        CVaR = np.mean(r[0:num_below_conf])

        # update the security attribute
        self.CVaR = CVaR
        return CVaR


    def calc_max_drawdown(self):
        """
        Maximum Draw Down
        How much would you loose if you bought at the highest point and
        sold at the lowest point?
        """
        # cumulive returns
        cumulative_returns = np.cumprod(1 + np.array(self.returns))

        # get the max cumulive return at each point in the time-series
        max_cumulative_return = np.maximum.accumulate(cumulative_returns)

        # calc the drawdown at each point in the series
        drawdown = cumulative_returns / max_cumulative_return - 1

        # find the worst drawdown
        max_drawdown = np.min(drawdown)

        # update the security attriubte
        self.MDD = max_drawdown

        return max_drawdown


    def generate_returns(self, mu, sigma, n):
        # init an array of n returns from the normal dist with mean=mu and stdev = sigma
        self.returns = np.random.default_rng().normal(mu, sigma, n)

    def get_price_data(self,key,from_year=None):
        """
        Returns daily data for a stock (symbol)
        key: api key
        symbols: GLD(gold),SPY(S&P500), LQD (iShares Corp Bond),VTC (vanguard total return bond)
        """
        url = f"https://www.alphavantage.co/query?function=TIME_SERIES_MONTHLY&symbol={self.identifier}&apikey={key}"
        r = requests.get(url)
        d = r.json()

        try:
            # check if data was returned
            df = pd.DataFrame(d['Monthly Time Series']).T
        except:
            print(f"Error for {self.identifier}:{d}")
            self.prices = []

        # extract data to a df
        df.columns = ['open','high','low','close','volume']
        df['symbol'] = self.identifier

        # change data types
        df.index = pd.to_datetime(df.index)
        df = df.sort_index()

        # convert datatype to float
        for col in ['open','high','low','close','volume']:
            df[col] = df[col].astype('float')

        # calc monthly returns
        df['returns'] = df['close'].pct_change()

        # filter by year if provided
        if from_year is not None:
            df = df.query('index>=@from_year')

        # calc mean & stdev (annualized)
        m = (1+np.mean(df['returns']))**12 -1
        s = np.std(df['returns']) * np.sqrt(12)

        self.prices = df
        self.returns = df['returns']
        self.mu = m
        self.sigma = s
        self.sharpe_ratio = self.calc_sharpe_ratio(rf=self.rf)
        self.MDD = self.calc_max_drawdown()
        self.CVaR = self.calc_CVaR(conf_level = self.conf_level)

    def plot_prices(self):
        df = self.prices
        hover_temp = "Date:%{x}"+ "Closing Price:%{y:,.0f}"
        fig = go.Figure(
            go.Scatter(
                x = df.index,
                y = df.close,
                line = dict(color = 'rgb(83,128,141)',width=2),
                fill='tozeroy',
                fillcolor = 'rgba(83,128,141,0.5)',
                hovertemplate = hover_temp
            )
        )
        fig.update_layout(
            template='plotly_white',
            title = f"{self.name} ({self.identifier})",
            yaxis_title='Closing Price' ,
            width=500, height=500
        )
        fig.show()

class Equity(Security): # a type of security
    def __init__(self, name, identifier, mu, sigma, n, rf, conf_level):
        Security.__init__(self, name, identifier,mu, sigma, n, rf, conf_level)
        self.type = 'equity'


class Fixedincome(Security): # a type of security
    def __init__(self, name, identifier, mu, sigma, n, rf, conf_level, yld):
        Security.__init__(self, name, identifier,mu, sigma, n, rf, conf_level)
        self.type = 'fixed income'
        self.yld = yld

    def __str__(self):
        print(Security.__str__(self))
        return f"Yield:{self.yld*100}%"


class Option(Security): # a type of security
    def __init__(self, name, identifier, mu, sigma, n, rf, conf_level,type, strike_price, expiration_date):
        Security.__init__(self, name, identifier, mu, sigma, n, rf, conf_level)
        self.type = type # put/call
        self.strike_price = strike_price
        self.expiration_date = expiration_date

    def __str__(self):
        print(Security.__str__(self))
        return f"Option Type:{self.type}\nStrike Price:{self.strike_price}\nExpiration Date:{self.expiration_date}"


class Future(Security): # a type of security
    def __init__(self, name, identifier, mu, sigma, n, rf, conf_level,type, direction, settlement_price, expiration_date, notional_amount):
        Security.__init__(self, name, identifier, mu, sigma, n, rf, conf_level)
        self.type = type # futures, forward
        self.direction = direction # long, short
        self.settlement_price = settlement_price
        self.expiration_date = expiration_date
        self.notional_amount = notional_amount

    def __str__(self):
        print(Security.__str__(self))
        return f"Futures Type:{self.type}\nDirection:{self.direction}\nSettlment Price:{self.settlement_price}\nExpiration Date:{self.expiration_date}\nNotional Amount:{self.notional_amount}"


# **Portfolios**

In [4]:
class Portfolio:
    def __init__(self ,name, securities, target_weights, portfolio_value, rf, conf_level):
        self.name = name
        self.securities = securities
        self.target_weights = target_weights
        self.portfolio_value = portfolio_value
        self.rf = rf
        self.conf_level = conf_level
        self.cov = None
        self.simulation_results = None
        self.mean_return = None
        self.mean_volatility = None
        self.sharpe_ratio = None
        self.expected_values = None
        self.CVaR = None
        self.MDD = None

    def __str__(self):
        msg = ""
        msg += f"{self.name}\n"
        msg += "="*75 + "\n"
        msg += f"Porfolio Value: ${self.portfolio_value:,.0f}\n"

        if self.mean_return is not None:
            msg += f"Mean Return: {self.mean_return*100:.2f}% (annualized)\n"
        if self.mean_volatility is not None:
            msg += f"Mean Volatility: {self.mean_volatility*100:.2f}% (annualized)\n"
        if self.sharpe_ratio is not None:
            msg += f"Sharpe Ratio: {self.sharpe_ratio}\n"
        if self.CVaR is not None:
            msg += f"CVar: {self.CVaR*100:.02f}%\n"
        if self.MDD is not None:
            msg += f"Max Drawdown: {self.MDD*100:.02f}%\n"

        # get holdings & weights
        msg += "Holdings\n"
        securities = ""
        for i, sec in enumerate(self.securities):
            if self.target_weights[i]>0:
                securities += f"=>{sec.name}:{self.target_weights[i]*100}% \n"
        msg += securities

        return msg

    def calc_sharpe_ratio(self):
        return round((self.mean_return  - self.rf) / self.mean_volatility,2)

    def calc_CVaR(self, returns, conf_level):
        """
        conditional value at risk for a confidence interval
        Find the average of the worst returns (below the conf level)
        """
        # number of returns
        n = len(returns)

        # sort the returns
        r = np.sort(returns)

        # number of returns lower than the confidence level
        num_below_conf = int((1- conf_level) * n)

        # average of the worst returns (below the conf level)
        CVaR = np.mean(r[0:num_below_conf])

        # update the security attribute
        self.CVaR = CVaR
        return CVaR


    def calc_max_drawdown(self, returns):
        """
        Maximum Draw Down
        How much would you loose if you bought at the highest point and
        sold at the lowest point?
        """
        # cumulive returns
        cumulative_returns = np.cumprod(1 + np.array(returns))

        # get the max cumulive return at each point in the time-series
        max_cumulative_return = np.maximum.accumulate(cumulative_returns)

        # calc the drawdown at each point in the series
        drawdown = cumulative_returns / max_cumulative_return - 1

        # find the worst drawdown
        max_drawdown = np.min(drawdown)

        # update the security attriubte
        self.MDD = max_drawdown

        return max_drawdown

    def calc_covariance(self):
        for i,sec in enumerate(self.securities):
            r = pd.DataFrame(sec.prices['returns'])
            r.columns = [f'returns_{sec.identifier}']
            if i==0:
                df_r = r
            else:
                df_r = df_r.join(r, how='inner',lsuffix=f'_{sec.identifier}')
        # update the cov matrix for the portfolio
        self.cov = df_r.cov()


    def convert_from_annual(self,metric,freq,measure):
        f = {
            'annual':1,
            'quarterly':4,
            'monthly':12,
            'daily':250
        }
        if measure=='mean':
            m = (1+metric)**(1/f[freq])-1
        elif measure=='std':
            m = metric / np.sqrt(f[freq])
        else:
            m = None
        return m

    def convert_to_annual(self, metric, freq, periods,measure):
        f = {
            'annual':1,
            'quarterly':4,
            'monthly':12,
            'daily':250
        }
        if measure == 'mean':
            m = (1+metric)**(f[freq]/periods)-1
        elif measure=='std':
            m = metric * np.sqrt(f[freq])
        else:
            m=None
        return m

    def get_security_prices(self,yr, key, plot=False,requests_per_min=5):
        """
        Get prices for each security in the portfolio from alphavantage
        """
        n_sec = len(self.securities)
        if n_sec == 0:
            return None

        # determine the throttle for API requests
        total_mins = ceil(n_sec / requests_per_min)
        throttle = (total_mins / n_sec)*60

        for sec in self.securities:
            # get the security prices from the alphavantage API
            sec.get_price_data(key, yr)

            if plot: # optional ploting
                sec.plot_prices()

            # pause execution to stay without API limits
            sleep(throttle)

    def get_expected_values(self):
        # extract portfolio values for each sim
        vals = []
        for sim,v in self.simulation_results.items():
            vals.append(np.array(v['portfolio-values']))
        vals = np.array(vals).T

        # calc the upper/lower conf intervals for each period in the simulation
        p = {
            'mean':[],
            'sig1-upper':[],
            'sig1-lower':[],
            'sig2-upper':[],
            'sig2-lower':[],
            'sig3-upper':[],
            'sig3-lower':[]
        }
        for i in range(len(vals)):
            sig,m, = np.std(vals[i]), np.mean(vals[i])
            p['mean'].append(m)
            p['sig1-upper'].append(m+sig)
            p['sig1-lower'].append(m-sig)
            p['sig2-upper'].append(m+sig*2)
            p['sig2-lower'].append(m-sig*2)
            p['sig3-upper'].append(m+sig*3)
            p['sig3-lower'].append(m-sig*3)

        # return a df
        return pd.DataFrame(p)

    def plot_boxplots(self):
        # get the return/risk values from the simulation
        pr = list({k:v['portfolio-total-return']*100 for k,v in self.simulation_results.items()}.values())
        pv = list({k:v['portfolio-std']*100 for k,v in self.simulation_results.items()}.values())
        cvar = list({k:v['portfolio-cvar']*100 for k,v in self.simulation_results.items()}.values())
        mdd = list({k:v['portfolio-mdd']*100 for k,v in self.simulation_results.items()}.values())

        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles = ['Returns (% Annual)', 'Volatility (% Annual)', 'Conditional VaR (%)', 'Max Drawdown (%)'],
            shared_yaxes=True
        )

        fig.add_trace(
            go.Box(
                y=pr,
                name="Return(%)",
                boxpoints='all',
                marker_color='rgba(83,128,141,0.4)',
                line_color='rgb(83,128,141)'
            ),row=1, col=1
        )
        fig.add_trace(
            go.Box(
                y=pv,
                name="Risk(%)",
                boxpoints='all',
                marker_color='rgba(83,128,141,0.4)',
                line_color='rgb(83,128,141)'
            ),row=1, col=2
        )
        fig.add_trace(
            go.Box(
                y=cvar,
                name="CVaR(%)",
                boxpoints='all',
                marker_color='rgba(83,128,141,0.4)',
                line_color='rgb(83,128,141)'
            ),row=2, col=1
        )
        fig.add_trace(
            go.Box(
                y=mdd,
                name="Max Drawdown(%)",
                boxpoints='all',
                marker_color='rgba(83,128,141,0.4)',
                line_color='rgb(83,128,141)'
            ),row=2, col=2
        )
        fig.update_layout(template='plotly_white',width=700, height=900,showlegend=False,title="Return/Risk Distributions")
        fig.show()


    def plot_histograms(self):
        pr = list({k:v['portfolio-total-return']*100 for k,v in self.simulation_results.items()}.values())
        pv = list({k:v['portfolio-std']*100 for k,v in self.simulation_results.items()}.values())

        # on-hover template
        hover_temp_return = "Return Range:%{x}"+ "Frequency:%{y:,.0f}"
        hover_temp_risk = "Volatility Range:%{x}"+ "Frequency:%{y:,.0f}"

        fig = make_subplots(rows=1,cols=2,subplot_titles = ['Annualized Returns (%)', 'Annualized Volatility (%)'],shared_xaxes=True, shared_yaxes=True)
        fig.add_trace(go.Histogram(x=pr,name='Returns',marker_color = 'rgba(83,128,141,0.8)',hovertemplate=hover_temp_return),row=1,col=1)
        fig.add_trace(go.Histogram(x=pv,name='Volatility',marker_color='rgba(83,128,141,0.5)',hovertemplate=hover_temp_risk),row=1,col=2)
        fig.update_layout(template='plotly_white',title_text='Histograms', width=700, height=500)
        fig.show()

    def plot_portfolio_simulations(self):
        sims = len(self.simulation_results)
        freq = {'annual':'Years','monthly':'Months','quarterly':'Quarters','daily':'Days'}
        title = f"Growth of ${self.portfolio_value:,.0f}: (Return:{round(self.mean_return*100,2)}%, Volatility:{round(self.mean_volatility*100,2)}%)Number of Simulations:{sims:,.0f}"

        # on-hover template
        hover_temp = "Simiulation:%{x}"+ "Porfolio Value:$%{y:,.0f}"

        # figure
        fig = go.Figure()
        for sim,v in self.simulation_results.items():
            fig.add_trace(
                go.Scatter(
                    x = list(range(len(v['portfolio-values']))),
                    y = v['portfolio-values'],
                    name = f'Simulation:{sim+1}',
                    hovertemplate = hover_temp,
                    marker = dict(color='rgba(83,128,141,0.5)')
                )
            )
        fig.update_layout(
            template='plotly_white',
            title = title,
            yaxis_title='Portfolio Value',
            xaxis_title=f"{freq[self.simulation_results[0]['frequency']]}",
            width=700,
            height=500,
            showlegend=False
        )
        fig.show()


    def plot_expected_values(self):
        freq = {'annual':'Years','monthly':'Months','quarterly':'Quarters','daily':'Days'}
        bands = {
            'sig1':{'upper':{'series':'sig1-upper','name':'1-Sigma (68%)'},
                    'lower':{'series':'sig1-lower','name':'1-Sigma (68%)'},
                    'color':'rgba(83,128,141,0.5)'
                   },
            'sig2':{'upper':{'series':'sig2-upper','name':'2-Sigma (95%)'},
                    'lower':{'series':'sig2-lower','name':'2-Sigma (95%)'},
                    'color':'rgba(83,128,141,0.3)'
                   },
            'sig3':{'upper':{'series':'sig3-upper','name':'3-Sigma (99%)'},
                    'lower':{'series':'sig3-lower','name':'3-Sigma (99%)'},
                    'color':'rgba(83,128,141,0.1)'
                   }
        }
        # get error bands
        df = self.expected_values

        # on-hover template
        hover_temp = "Period:%{x}"+ "Porfolio Value:$%{y:,.0f}"

        # figure
        fig = go.Figure()
        # error bands
        for k,v in bands.items():
            # upper
            fig.add_trace(
                go.Scatter(
                    name=v['upper']['name'],
                    x = df.index,
                    y = df[v['upper']['series']],
                    mode='lines',
                    line=dict(width=0),
                    showlegend=False,
                    hovertemplate = hover_temp
                )
            )
            # lower
            fig.add_trace(
                go.Scatter(
                    name=v['lower']['name'],
                    x = df.index,
                    y = df[v['lower']['series']],
                    mode='lines',
                    line=dict(width=0),
                    fill='tonexty',
                    fillcolor = v['color'],
                    showlegend=True,
                    hovertemplate = hover_temp
                )
            )
        # mean
        fig.add_trace(
            go.Scatter(
                x = df.index,
                y = df['mean'],
                name = 'mean',
                mode = 'lines',
                fill = None,
                line = dict(width=3, color='#FC4C01'),
                hovertemplate = hover_temp
            )
        )
        # upper and lower portfolio values at 1-sigma
        lower = df.at[df.shape[0]-1,'sig1-lower']
        upper = df.at[df.shape[0]-1,'sig1-upper']
        freq = freq[self.simulation_results[0]['frequency']]
        periods = len(self.simulation_results[0]['portfolio-values'])-1
        title_1 = f"Expected Growth of a ${self.portfolio_value:,.0f} Portfolio after {periods} {freq} "
        title_2 = f"Ending Value:${lower:,.0f} to {upper:,.0f} (68% Confidence) "
        title_3 = f"Sharpe Ratio:{self.sharpe_ratio:,.2f} | Annual Return: {self.mean_return*100:,.2f}%"

        fig.update_layout(
            template='plotly_white',
            title=title_1 + title_2 + title_3,
            xaxis_title=freq,
            yaxis_title='portfolio value',
            width=700,
            height=500,
            legend=dict(yanchor='top', y=0.97, xanchor='left', x=0.03)
        )
        fig.show()


    def run_simulation(self, simulations, years, frequency,robust=False):
        """
        years: number of years in the simulation
        frequency: annual, quarterly,monthly, daily
        simulations: number of simulations
        """
        # calc the number of periods
        freq = {'annual':1, 'quarterly':4,'monthly':12,'daily':220}
        periods = years * freq[frequency]

        # results accumulator
        results = {}

        # initial holdings
        portfolio = np.array([w*self.portfolio_value for w in self.target_weights])

        if robust:
            print("Monte Carlo Simulation")
            print("="*100)
        for sim in range(simulations):
            returns = []  # security returns
            s_values = [] # security values
            csr = []      # cumulative security return
            s_std = []    # security return standard deviation
            result = {    # simulation results
                'security-returns':[],
                'security-values':[],
                'security-total-return':[],
                'security-std':None,
                'portfolio-returns':[],
                'portfolio-values':[],
                'portfolio-total-return':None,
                'portfolio-std':None,
                'portfolio-cvar': None,
                'portfolio-mdd': None,
                'frequency':frequency
            }

            # simulate multivariate returns for each security
            mus = [self.convert_from_annual(m.mu, frequency, 'mean') for m in self.securities]
            rets = np.random.default_rng().multivariate_normal(mus, np.array(self.cov), periods).T

            for i,sec in enumerate(self.securities):
                returns.append(rets[i])
                s_std.append(np.std(rets[i]))
                sec.returns = rets[i]
                s_returns = 1 + sec.returns
                csr.append(s_returns.prod()-1)
                s_values.append(np.cumprod(s_returns)*portfolio[i])


            # Portfolio Calculations
            # transform the security returns
            returns = np.array(returns).T

            # calc the weighted portfolio returns
            p_returns = returns.dot(self.target_weights)

            # calc portfolio value
            p_values = np.cumprod(p_returns + 1)*self.portfolio_value

            # cumulative portfolio return
            cpr = (p_returns + 1).prod()-1

            # portfolio standard deviation
            p_std = np.std(p_returns)

            # portfolio CVaR (annualized)
            CVaR = self.calc_CVaR(conf_level=self.conf_level, returns=p_returns)

            # portfolio max drawdown (annualized)
            MDD = self.calc_max_drawdown(returns=p_returns)

            # annualize portfoio return, volatility
            r = self.convert_to_annual(cpr, frequency, periods,'mean')
            v = self.convert_to_annual(p_std, frequency, periods,'std')

            # accumulate data summaries
            result['security-returns'] = returns
            result['security-values'] = s_values
            result['security-total-return'] = csr
            result['security-std'] = s_std
            result['portfolio-returns'] = p_returns
            result['portfolio-total-return']= r
            result['portfolio-values'] = np.insert(p_values,0,self.portfolio_value)
            result['portfolio-std'] = v
            result['portfolio-cvar'] = CVaR
            result['portfolio-mdd'] = MDD

            # accumulate simulation summary
            results[sim] = result

        # simulation statisitics (portfolio level, aggregate over all simulations)
        pr = round(np.mean(list({k:v['portfolio-total-return'] for k,v in results.items()}.values())),4)
        pv = round(np.mean(list({k:v['portfolio-std'] for k,v in results.items()}.values())),4)
        cvar = round(np.mean(list({k:v['portfolio-cvar'] for k,v in results.items()}.values())),4)
        mdd = round(np.mean(list({k:v['portfolio-mdd'] for k,v in results.items()}.values())),4)

        # save results as attributes in the portfolio class
        self.simulation_results = results
        self.mean_return = pr
        self.mean_volatility = pv
        self.sharpe_ratio = self.calc_sharpe_ratio()
        self.expected_values = self.get_expected_values()
        self.CVaR = cvar
        self.MDD = mdd


    def weight_combinations(self, total_portfolio_wt, min_security_wt, max_security_wt, weight_increment):
        """get all portfolio weight combinations"""

        # get the number of securities in the portfolio
        n_securities = len(self.securities)

        # check the max security weight is large enough to satisify the total portfolio weight
        check = n_securities * max_security_wt >= total_portfolio_wt

        if check:
            # get list of possible security weights
            weights = list(range(min_security_wt,max_security_wt+weight_increment,weight_increment))

            # get list of all combinations of security weights
            possible_weights = itertools.product(weights, repeat = n_securities - 1)

            sim_weights =[]
            for perm in possible_weights:
                final = total_portfolio_wt - sum(perm)
                if final in weights:
                    # if weights sum to the target weight
                    # convert to decimal, accumulate
                    wts = perm + (final,)
                    sim_weights.append([w/100 for w in wts])
        else:
            sim_weights = []
            print("Max individual security weight too low for the number of securities in the portfolio. Increase max security weight")
        return  sim_weights

    def build_efficient_frontier(self, total_portfolio_wt, min_security_wt, max_security_wt, weight_increment, num_sims, years, frequency, verbose=0):
        """ """

        # get all possible combinations of weights that add to the target weight
        weights = self.weight_combinations(total_portfolio_wt, min_security_wt, max_security_wt, weight_increment)

        num_eff_frontier_sims = len(weights)
        if num_eff_frontier_sims==0:
            print("=>No simulations will run")
            return None

        # frontier results
        results ={
            'simulation':[],
            'expected-return':[],
            'expected-risk':[],
            'sharpe-ratio':[],
            'weights':[],
            'cvar':[],
            'mdd':[]
        }
        if verbose>0:
            # run simulations for each possible security weights
            print(f"Efficient Frontier Simulations:{len(weights):,.0f} Possible Portfolios")
            print("="*100)

        # run simulations for each possible security weights
        for sim, weight in enumerate(weights):
            if verbose>0 and sim%verbose==0:
                progress = round(sim/num_eff_frontier_sims,2)
                print(f"=>{progress*100:,.0f}% complete")

            self.target_weights = weight
            self.run_simulation(simulations=num_sims,years=years,frequency=frequency)
            results['simulation'].append(sim)
            results['expected-return'].append(self.mean_return)
            results['expected-risk'].append(self.mean_volatility)
            results['sharpe-ratio'].append(self.sharpe_ratio)
            results['weights'].append(weight)
            results['cvar'].append(self.CVaR)
            results['mdd'].append(self.MDD)

        # add results to the portfolio
        self.efficient_frontier = results

        # update the portfolio target weights with the most optimal weights
        self.target_weights = self.get_optimal_portfolios(1)['weights'].item()

    def get_optimal_portfolios(self, top_n):
        try:
            df = pd.DataFrame(self.efficient_frontier)
            df.sort_values(by='sharpe-ratio',inplace=True, ascending=False)
            return df.iloc[0:top_n,]
        except:
            return None

    def plot_efficient_frontier(self):
        """ """
        # get the effient frontier results
        df = pd.DataFrame(self.efficient_frontier)

        # convert to return/risk to % scale
        df['expected-risk']=df['expected-risk']*100
        df['expected-return']=df['expected-return']*100

        # get axes ranges +/- 2 percent
        x_min = min(df['expected-risk'])-2
        x_max = max(df['expected-risk'])+2
        y_min = min(df['expected-return'])-2
        y_max = max(df['expected-risk'])+2

        # get weight labels (concate the weights)
        weight_label = []
        for weights in self.efficient_frontier['weights']:
            w = [f"{round(w*100,2)}%" for w in weights]
            w = "|".join(w)
            weight_label.append(w)
        df['portfolio-weights'] = weight_label

        # get security names (concate the names)
        secs = ""
        for sec in self.securities:
            secs += sec.identifier + "|"

        # build plot title
        title = "Portfolio Risk/Reward Simulation"
        title_2 = secs[:len(secs)-1]

        # build the plot
        fig = px.scatter(
            df,
            x='expected-risk',
            y='expected-return',
            hover_data = ['portfolio-weights','sharpe-ratio'],
            color = 'sharpe-ratio',
            opacity = 0.6,
            color_continuous_scale=px.colors.diverging.Earth
        )

        # adjust the plot
        fig.update_traces(marker_size=14)
        fig.update_layout(template='plotly_white',width=700,height=500,title=title+title_2,legend_title='Portfolio Weights',hovermode="y")
        fig.update_xaxes(range=[x_min,x_max],title='risk (%)')
        fig.update_yaxes(range=[y_min,y_max],title='return (%)')
        fig.show()

# **Setup Securities**

In [5]:
# get securities
s1 = Equity(name='SPDR S&P 500 ETF Trust',identifier='SPY', mu= 0.05, sigma=0.05, n=100, rf=RF, conf_level=CONF_LEVEL)
s2 = Equity(name='iShares MSCI EAFE',identifier='IEFA', mu= 0.05, sigma=0.05, n=100, rf=RF, conf_level=CONF_LEVEL)
s3 = Equity(name='iShares MSCI Min Vol Emerging Markets', identifier='EEMV',mu=0.08,sigma=0.16,n=100, rf=RF, conf_level=CONF_LEVEL)
s4 = Equity(name='Mackenzie Canadian Equity Index', identifier='TSE:QCN',mu=0.08,sigma=0.16,n=100, rf=RF, conf_level=CONF_LEVEL)
s5 = Fixedincome(name='BMO Long Federal Bond Index', identifier='TSE:ZFL',mu=0.08,sigma=0.16,n=100,yld=0.0, rf=RF, conf_level=CONF_LEVEL)
s6 = Fixedincome(name='BMO Aggregate Bond Index', identifier='TSE:ZAG',mu=0.08,sigma=0.16,n=100,yld=0.0, rf=RF, conf_level=CONF_LEVEL)
s7 = Fixedincome(name='BlackRock/iShares Core Cnd Short Term Corp', identifier='TSE:XSH',mu=0.08,sigma=0.16,n=100,yld=0.0, rf=RF, conf_level=CONF_LEVEL)
s8 = Equity(name='Gold ETF', identifier='GLD',mu=0.08,sigma=0.16,n=100, rf=RF, conf_level=CONF_LEVEL)

# **Build the Portfolio**

In [6]:
# build the portfolio
p = Portfolio(
    name='Portfolio:RRSPs',
    securities = [s1,s2,s3,s4,s5,s6,s7,s8],
    target_weights =  [1/8]*8, # start with even weights
    portfolio_value=100000,
    rf = RF,
    conf_level = CONF_LEVEL
)

# get prices for each security in the portfolio from alpha vantage
p.get_security_prices(yr=2013, key=key, plot=False,requests_per_min=5)

# calculate the covariance between securities in the portfolio
p.calc_covariance()

# **Determine the Optimal Security Weights in the Portfolio**

It's a good idea to check the total portfolio combinations before running the simulation. The weight_combinations method in the portfolio class can be used to determine the number of possible combinations of portfolio weights based on the weight contraints. Check this lenght of the output before trying to build the efficient frontier - Some combinations will never return results!

*   **total_portfolio_wt** = The total sum of all weights. For long portfolios without leverage, the weights should add to 100
* **min_security_wt** = The minimum weight of each security. A setting of zero means the securities can be excluded from the portfolio
* **max_security_wt** = The maximum weight each security can occupy in the portfolio
* **weight_increment** = The spacing between security weights (ie,a value of 10 means each security weight will be in increments of 10%, 20%, 30% etc). The more securities you add, the larger the increment should be, otherwise there will be an impractical number of possible portfolios to model



In [7]:
len(p.weight_combinations(total_portfolio_wt=100, min_security_wt=0, max_security_wt=30, weight_increment=10))

6728

# **Build the Efficient Frontier from all possible weight combinations**

In [8]:
# build the efficient frontier
p.build_efficient_frontier(
    total_portfolio_wt=100,
    min_security_wt=0,
    max_security_wt=30,
    weight_increment=10,
    num_sims=100,
    years=10,
    frequency='monthly',
    verbose = 1000  # print progress every 1,000 portfolios
)
p.plot_efficient_frontier()
p.get_optimal_portfolios(5)

Efficient Frontier Simulations:6,728 Possible Portfolios
=>0% complete
=>15% complete
=>30% complete
=>45% complete
=>59% complete
=>74% complete
=>89% complete


,simulation,expected-return,expected-risk,sharpe-ratio,weights,cvar,mdd
5870,5870,0.0940,0.1205,0.45,"[0.3, 0.0, 0.1, 0.3, 0.0, 0.0, 0.0, 0.3]",-0.0629,-0.2025
5839,5839,0.0856,0.1096,0.42,"[0.3, 0.0, 0.1, 0.2, 0.0, 0.0, 0.1, 0.3]",-0.0567,-0.1755
5727,5727,0.0886,0.1144,0.42,"[0.3, 0.0, 0.0, 0.3, 0.0, 0.1, 0.0, 0.3]",-0.0595,-0.1920
6463,6463,0.0940,0.1274,0.42,"[0.3, 0.2, 0.0, 0.2, 0.0, 0.0, 0.0, 0.3]",-0.0664,-0.2128
6172,6172,0.0871,0.1184,0.40,"[0.3, 0.1, 0.0, 0.3, 0.0, 0.0, 0.1, 0.2]",-0.0620,-0.2025


# **Run a Monte Carlo Simulation for the Most Efficient Portfolio**

In [9]:
# run a monte carlo simulation with the updated target weights
p.run_simulation(simulations=500,years=10,frequency='monthly')
p.plot_portfolio_simulations()
p.plot_boxplots()
p.plot_expected_values()
print(p)

Portfolio:RRSPs
Porfolio Value: $100,000
Mean Return: 9.04% (annualized)
Mean Volatility: 12.09% (annualized)
Sharpe Ratio: 0.42
CVar: -6.32%
Max Drawdown: -19.88%
Holdings
=>SPDR S&P 500 ETF Trust:30.0% 
=>iShares MSCI Min Vol Emerging Markets:10.0% 
=>Mackenzie Canadian Equity Index:30.0% 
=>Gold ETF:30.0% 

